# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1.
2.
3.
4.
5.

1) 5 и (2, 3): элемент во второй строке и третьем столбце равен 5,shape показывает размер массива

2) 1 и 8: argmax() возвращает индекс максимального элемента первичного максимума, а max() — само значение

3) 99: срез a[::3] обычно является представлением (view) исходного массива, поэтому изменение элемента среза меняет a

4) 2 и 0.5: условие истинно для двух из четырёх элементов, среднее булевой маски равно доле истинных значений

5) (2, 3) и 0.25: zeros создаёт массив указанной формы, а второй элемент linspace(0, 1, 5) равен 0.25

In [52]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [53]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [74]:
# База
#1
i_max=amount.argmax()
print("Позиция:",i_max)
print(f"Сумма:{amount[i_max]:,.0f}, клиент:{cust[i_max]}, город:{city[i_max]}")


Позиция: 1001
Сумма:9,999,990, клиент:TEST, город:Алматы


Для менеджера: Стоит перепроверить корректность данных, так как возможно это тестовые заказы

In [76]:
#2
paid_sum = amount[paid].sum()
not_paid_sum = amount[~paid].sum()
print("Оплаченные заказы:", f"{paid_sum:,.0f}")
print("Неоплаченные заказы:", f"{not_paid_sum:,.0f}")

Оплаченные заказы: 101,909,360
Неоплаченные заказы: 7,974,000


Для менеджера: Оплаченная выручка это фактически полученные деньги, неоплаченные суммы в неё не входят

In [77]:
#3
shymkent_mask = city == "Шымкент"
print(f"Заказов в Шымкенте: {shymkent_mask.sum()} из {len(city)}")
print(f"Доля: {shymkent_mask.mean():.2%}")

Заказов в Шымкенте: 804 из 5463
Доля: 14.72%


Для менеджера: примерно 15% всех заказов приходится на Шымкент

## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [82]:
#4
discount_mask = paid & (amount > 50_000)
discount_all = amount[discount_mask].sum() * 0.15
discount_without_test = amount[discount_mask & (cust != "TEST")].sum() * 0.15
print(f"Со всеми заказами: {discount_all:,.0f}, без TEST: {discount_without_test:,.0f}")

Со всеми заказами: 5,702,422, без TEST: 1,202,427


Для менеджера: исключение TEST резко уменьшает оценку скидки, потому что самый дорогой заказ на 9 999 990 ₸ принадлежит TEST

In [78]:
#5
print("Доля доставок за 1–2 дня:")
for c in np.unique(city):
    m = city == c
    share = ((days[m] >= 1) & (days[m] <= 2)).sum() / m.sum()
    print(f"   {c}: {share:.2%}")

Доля доставок за 1–2 дня:
   Актобе: 62.22%
   Алматы: 64.53%
   Астана: 63.53%
   Атырау: 64.38%
   Караганда: 67.84%
   Шымкент: 29.85%


Для менеджера: в большинстве городов доля быстрой доставки около 62–68%. В отличии от других городов Шымкент заметно отстает

In [79]:
#6
# Ошибка 1: среднее считается по всем заказам, включая неоплаченные
# Ошибка 2: round(avg) округляет сумму до целого и скрывает точность результата
avg = amount[paid].mean()
print(round(avg, 2))



20780.86


Для менеджера: средний чек по оплаченным заказам составляет около 20,8 тыс

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [83]:
# Челлендж
#7
import timeit
def sum_with_loop():
    total = 0.0
    for value, is_paid in zip(amount, paid):
        if is_paid:
            total += value
    return total
def sum_with_mask():
    return amount[paid].sum()

loop_time = min(timeit.repeat(sum_with_loop, number=100, repeat=5))
mask_time = min(timeit.repeat(sum_with_mask, number=100, repeat=5))
print(f"Цикл: {loop_time:.5f} с, маска NumPy: {mask_time:.5f} с, ускорение: {loop_time / mask_time:.1f}")

Цикл: 0.08109 с, маска NumPy: 0.00165 с, ускорение: 49.2


Для менеджера: NumPy-маска быстрее, потому что операции над массивами выполняются оптимизированным кодом без Python-цикла по каждому заказу

In [86]:
#8
np.random.seed(42)
group_a = (np.random.rand(1000) < 0.05).sum()
group_b = (np.random.rand(1000) < 0.06).sum()
print(f"Первый эксперимент: A = {group_a} покупок, B = {group_b} покупок")
b_better = 0
for _ in range(100):
    a = (np.random.rand(1000) < 0.05).sum()
    b = (np.random.rand(1000) < 0.06).sum()
    b_better += b > a
print(f"В 100 повторениях группа B лучше группы A в {b_better} случаях")

Первый эксперимент: A = 54 покупок, B = 58 покупок
В 100 повторениях группа B лучше группы A в 84 случаях


Для менеджера: 1)в первом запуске B получила больше покупок. Но даже при реальном росте конверсии с 5% до 6%

2)случайность выборки иногда меняет результат. 1 000 клиентов на группу может быть недостаточно для уверенного вывода

3)для решения о запуске тест стоит дополнить проверкой статистической значимости и увеличить выборку